# AirSense - 03 API Demo
A tour of every endpoint using plain `requests`: exactly what the Streamlit UI does behind the scenes, and a quick way to check the API contract in `docs/LLD.md`.

**Start the API first** (from the project root): `uvicorn backend.main:app --port 8000`
Change the address with the `API_URL` environment variable.

In [1]:
import io, os, time, json
import requests
import numpy as np, pandas as pd

API_URL = os.getenv("API_URL", "http://localhost:8000")

def show(resp, keys=None):
    body = resp.json()
    if keys: body = {k: body[k] for k in keys if k in body}
    print(resp.status_code, json.dumps(body, indent=2)[:900])

show(requests.get(f"{API_URL}/health"))

200 {
  "status": "ok",
  "model_loaded": true
}


## 1. Health and supported cities

In [2]:
cities = requests.get(f"{API_URL}/cities").json()["cities"]
print(len(cities), "cities:", cities)

10 cities: ['Ahmedabad', 'Aizawl', 'Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna']


## 2. A normal prediction
All six readings supplied. `confidence` is the model's top probability.

In [3]:
r = requests.post(f"{API_URL}/predict", json={"city": "Delhi", "date": "2026-11-02",
                  "pm25": 180, "pm10": 320, "no2": 60, "so2": 15, "co": 2.5, "o3": 40})
show(r, ["category", "confidence", "color", "headline", "advice", "imputed_fields", "low_confidence"])

200 {
  "category": "Very Poor",
  "confidence": 0.9996,
  "color": "#FF0000",
  "headline": "Air quality is very poor",
  "advice": "Respiratory illness on prolonged exposure. Avoid outdoor exercise.",
  "imputed_fields": [],
  "low_confidence": false
}


## 3. Blank readings are allowed (at least 2)
Missing readings are filled with the city's training median and **reported back** in `imputed_fields`, so nothing is assumed silently.

In [4]:
r = requests.post(f"{API_URL}/predict", json={"city": "Chennai", "pm25": 35, "pm10": 70})
show(r, ["category", "confidence", "imputed_fields"])

200 {
  "category": "Satisfactory",
  "confidence": 0.9593,
  "imputed_fields": [
    "NO2",
    "SO2",
    "CO",
    "O3"
  ]
}


## 4. Bad input never crashes the service
Each case returns a clear 4xx JSON error, never a 5xx or a stack trace.

In [5]:
cases = {
    "negative value":      {"city": "Delhi", "pm25": -5, "pm10": 80},
    "absurdly large":      {"city": "Delhi", "pm25": 99999, "pm10": 80},
    "CO in wrong units":   {"city": "Delhi", "pm25": 80, "pm10": 120, "co": 2500},
    "only one reading":    {"city": "Delhi", "pm25": 80},
    "unknown city":        {"city": "Atlantis", "pm25": 80, "pm10": 120},
    "text instead of number": {"city": "Delhi", "pm25": "abc", "pm10": 120},
    "empty body":          {},
}
rows = []
for name, payload in cases.items():
    r = requests.post(f"{API_URL}/predict", json=payload)
    b = r.json()
    rows.append({"case": name, "status": r.status_code, "error": b.get("error"), "message": b.get("message", "")[:70]})
pd.DataFrame(rows)

,case,status,error,message
0,negative value,422,validation_error,Some inputs are invalid.
1,absurdly large,422,validation_error,Some inputs are invalid.
2,CO in wrong units,422,validation_error,Some inputs are invalid.
3,only one reading,422,validation_error,Some inputs are invalid.
4,unknown city,422,unknown_city,City 'Atlantis' is not supported. Choose one o...
5,text instead of number,422,validation_error,Some inputs are invalid.
6,empty body,422,validation_error,Some inputs are invalid.


## 5. Batch prediction from CSV
We take the official template, then add two deliberately broken rows. Good rows are predicted; bad rows are reported **by CSV row number**.

In [6]:
template = requests.get(f"{API_URL}/template.csv").text
csv_text = template + "Atlantis,2026-11-02,50,90,,,,\nDelhi,2026-11-02,-3,90,,,,\nMumbai,2026-11-02,55,110,38,16,1.0,35\n"
print(csv_text)
r = requests.post(f"{API_URL}/predict/batch", files={"file": ("demo.csv", io.BytesIO(csv_text.encode()), "text/csv")})
out = r.json()
print(f"status {r.status_code}: total={out['total']} predicted={out['succeeded']} skipped={out['failed']}")
display(pd.DataFrame(out["results"])[["row", "city", "category", "confidence", "low_confidence"]])
display(pd.DataFrame(out["errors"]))

City,Date,PM2.5,PM10,NO2,SO2,CO,O3
Delhi,2026-11-02,145,290,62,18,2.1,41
Chennai,2026-11-02,32,70,24,11,0.8,30
Atlantis,2026-11-02,50,90,,,,
Delhi,2026-11-02,-3,90,,,,
Mumbai,2026-11-02,55,110,38,16,1.0,35

status 200: total=5 predicted=3 skipped=2


,row,city,category,confidence,low_confidence
0,2,Delhi,Very Poor,0.9947,False
1,3,Chennai,Satisfactory,0.9506,False
2,6,Mumbai,Moderate,0.9520,False


,row,reason
0,4,City 'Atlantis' is not supported. Choose one o...
1,5,Input should be greater than or equal to 0


In [7]:
# File-level problems are rejected with a clear message
for name, content in {"no City column": "PM2.5,PM10\n10,20\n", "header only": "City,PM2.5,PM10\n", "binary junk": b"\x00\x01\xff"}.items():
    data = content.encode() if isinstance(content, str) else content
    r = requests.post(f"{API_URL}/predict/batch", files={"file": ("x.csv", data, "text/csv")})
    print(f"{name:15s} -> {r.status_code} {r.json()['error']}: {r.json()['message']}")

no City column  -> 400 invalid_file: Missing required column 'City'.
header only     -> 400 invalid_file: The file has no data rows.
binary junk     -> 400 invalid_file: Could not read the file as CSV.


## 6. Model provenance and pipeline status
These power the **Pipeline** page in the UI.

In [8]:
info = requests.get(f"{API_URL}/model/info").json()
print({k: info[k] for k in ["model_name", "mlflow_run_id", "git_commit", "data_source", "trained_at"]})
print("test macro-F1:", info["test_metrics"]["macro_f1"], "| candidates (val macro-F1):", info["candidate_val_macro_f1"])
st = requests.get(f"{API_URL}/pipeline/status").json()
print("\nDAG edges:", st["dag"]["edges"], "\nRun summary:", st["summary"])
pd.DataFrame(st["history"])[["stage", "status", "seconds", "started_at"]]

{'model_name': 'random_forest', 'mlflow_run_id': 'a7a010613af3491bb0a5fd9fba1b181c', 'git_commit': 'a12c4086a37352d437f84bf7593c5d4730869851', 'data_source': 'synthetic', 'trained_at': '2026-11-07T07:32:02.910930+00:00'}
test macro-F1: 0.9644 | candidates (val macro-F1): {'logreg': 0.7813, 'random_forest': 0.958, 'hist_gb': 0.9456}



DAG edges: [['ingest', 'preprocess'], ['preprocess', 'train']] 
Run summary: {'success': 3, 'failed': 0, 'total': 3}


,stage,status,seconds,started_at
0,train,success,18.284,2026-11-07T07:31:44.637551+00:00
1,preprocess,success,9.702,2026-11-07T07:31:32.437614+00:00
2,ingest,success,0.234,2026-11-07T07:31:31.510870+00:00


## 7. Prometheus metrics
The same numbers Grafana plots.

In [9]:
text = requests.get(f"{API_URL}/metrics").text
for line in text.splitlines():
    if line.startswith(("airsense_predictions_total", "airsense_errors_total", "airsense_batch_rows_total", "airsense_model_loaded")):
        print(line)

airsense_errors_total{type="validation"} 6.0
airsense_errors_total{type="unknown_city"} 1.0
airsense_errors_total{type="invalid_file"} 3.0
airsense_predictions_total{category="Very Poor"} 2.0
airsense_predictions_total{category="Satisfactory"} 2.0
airsense_predictions_total{category="Moderate"} 1.0
airsense_batch_rows_total{outcome="ok"} 3.0
airsense_batch_rows_total{outcome="failed"} 2.0
airsense_model_loaded 1.0


## 8. Latency check
Sequential requests against the warm model. The acceptance criterion is p95 < 300 ms.

In [10]:
payload = {"city": "Delhi", "pm25": 150, "pm10": 280, "no2": 55}
times = []
for _ in range(50):
    t = time.perf_counter(); requests.post(f"{API_URL}/predict", json=payload); times.append((time.perf_counter() - t) * 1000)
print(f"p50 = {np.percentile(times, 50):.1f} ms | p95 = {np.percentile(times, 95):.1f} ms | max = {max(times):.1f} ms  (includes local HTTP)")

p50 = 10.1 ms | p95 = 11.0 ms | max = 11.3 ms  (includes local HTTP)
